In [12]:
import os
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from langchain_groq import ChatGroq

from langchain.tools import tool

from dotenv import load_dotenv

load_dotenv(override=True)

True

In [13]:
EMBEDDING_MODEL = os.getenv("ASKHR_EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2")
VECTOR_DB_DIR = os.getenv("ASKHR_VECTOR_DB_DIR", "AgenticHR/RAG_pipeline/PolicyVB")


llm = ChatGroq(model='llama-3.1-8b-instant')

In [14]:
_embeddings = None
_vector_db = None


def get_vector_db():
    global _embeddings, _vector_db
    if _vector_db is not None:
        return _vector_db
    _embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
    _vector_db = Chroma(
        persist_directory=VECTOR_DB_DIR,
        embedding_function=_embeddings,
    )

    return _vector_db


In [23]:
@tool
def retriever_tool(query: str, k: int = 5, search_type: str = "mmr") -> str:
    """
    Retrieve the content from the vector based on the query which can full fill the user query; act as you are hr
    """
    vector_db = get_vector_db()
    if vector_db is None:
        return "The knowledge base is temporarily unavailable. Please try again shortly."

    retriever = vector_db.as_retriever(
            search_type=search_type,
            search_kwargs={"k": k, "fetch_k": max(50, k * 5)},
        )
    docs = retriever.invoke(query)

    if not docs:
        return "No relevant documents found."

    return "\n\n".join(
        f"Source: {list(doc.metadata.get('source', 'Unknown').split('/'))[-1].split('.')[0]}\nContent: {doc.page_content}"
        for doc in docs
    )

TypeError: 'list' object is not callable

In [16]:
tool = [retriever_tool]

llm_with_tool = llm.bind_tools(tool)

In [17]:
reponse = llm_with_tool.invoke("what is the name of the company")

In [21]:
# 1. Ask the LLM (It decides to call the tool)
response = llm_with_tool.invoke("what is the name of the company")

# 2. Extract the tool arguments manually if not natively populated in .tool_calls
# (Since this model version is outputting text tags, we can check or pass directly)
print("🤖 LLM wants to call tool with:", response.content)

# 3. Manually execute your tool function with the query
tool_output = retriever_tool.invoke({"query": "what is the name of the company"})
print("🛠️ Tool Output (RAG Docs):\n", tool_output)

# 4. Pass the tool context back to the LLM along with the conversation history for the final answer
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

final_response = llm.invoke([
    HumanMessage(content="what is the name of the company"),
    response,  # The original tool call message from the LLM
    ToolMessage(content=tool_output, tool_call_id="manually_executed") # The RAG results
])

print("\n🤖 Final AI Answer:\n", final_response.content)

🤖 LLM wants to call tool with: 


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

🛠️ Tool Output (RAG Docs):
 Source: C:\Users\sinha\Documents\Project\Ask-HR\Company Info\data\departments
Content: department: Customer Success
headquarters: Bhubaneswar, Odisha, India

Source: employee_handbook
Content: **Company:** Feku Tech Solutions Pvt. Ltd.

Source: travel_policy
Content: behalf of the company for purposes such as:
* Client meetings
* Project implementation
* Business development activities
* Conferences and seminars
* Training programs
* Company-sponsored events
---
## Travel Authorization

Source: company_announcements
Content: ---
## Contact
For questions regarding company announcements, employees may contact:

Source: benefits_policy
Content: ## Learning & Development
The company encourages continuous learning through:

🤖 Final AI Answer:
 The name of the company is Feku Tech Solutions Pvt. Ltd.


In [22]:
print("\n🤖 Final AI Answer:\n", final_response.content)


🤖 Final AI Answer:
 The name of the company is Feku Tech Solutions Pvt. Ltd.
